# Stage 4c — Is the hand-size normalization hurting? (robust normalization experiment)

**Why this notebook.** In Stage 4b the plot of one clip showed normalized fingertip values of 80–90 "hand units". Our normalization divides every joint,
frame by frame, by the wrist-to-middle-finger-base distance *in the image plane*. When a hand points toward the camera that distance becomes tiny, and the values blow up.
A first check on the training clips gave: median absolute value 0.62, 90th percentile 1.69, 99th percentile 4.25, 99.9th percentile 14.3, and 0.7 % of all values above 5.
So the bulk is fine and the tail is small but heavy. This notebook measures whether repairing the tail helps the CNN.

**The four variants** (everything else identical: CNN, rotation augmentation, same folds):
| Variant | Hand-size scale | Clipping |
|---|---|---|
| `base` | wrist to middle-finger base, image plane (current) | none |
| `clip5` | current | every joint value limited to +/- 5 |
| `robust3d` | mean of the 3D distances from the wrist to the four finger bases (index, middle, ring, pinky), with z put in the same unit as x and y | none |
| `robust3d_clip5` | robust | +/- 5 |
The robust scale averages four bones and uses depth, so it should not collapse when one projected bone is short.
*Assumption to test, not a fact:* MediaPipe documents that z uses roughly the same scale as x; the experiment tells us whether using it helps.

**What the notebook does**
1. Builds the four sequence versions (cached on Drive) and unit-tests them.
2. Shows how the tails and the hand-size scale behave in each variant (Cell 5).
3. Trains the CNN + rotation recipe on each variant: 3 seeds on the single split, then 5-fold cross-validation, with paired comparisons against `base`.

**Expectation, honestly:** the tail is small, so I expect a small effect (zero to about 1-2 points). A null result is also useful: it tells us not to spend more time here.

**How to run:** a new Colab notebook with a **GPU runtime**; run the cells in order from Cell 2. Needs in Drive: `splits.csv`, `landmarks_raw.npz`, `frames_tracked_v2.npy`, and, if present, `results/stage4b_recipe.json` and `results/stage4b_results.csv`.
Rough time (an estimate, not measured): about 35-45 minutes. Results are saved after each phase. If a cell fails, send me the last lines of the error.

In [ ]:
# [setup] Cell 2 — configuration
from pathlib import Path
import json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/slovo_project')
except ImportError:                      # running outside Colab
    BASE = Path('./slovo_project')

PROC, FIG, RES = BASE / 'processed', BASE / 'figures', BASE / 'results'
for d in (FIG, RES):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
FIXED_T = 64
N_FOLDS = 5
FINAL_SEEDS = [0, 1, 2]
VARIANTS = ['base', 'clip5', 'robust3d', 'robust3d_clip5']
warnings.filterwarnings('ignore')
print('torch', torch.__version__, '| device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'no GPU')

In [ ]:
# Cell 3 — load the Stage 0 outputs (same files and same split as Stages 1 to 4)
splits = pd.read_csv(PROC / 'splits.csv')
z = np.load(PROC / 'landmarks_raw.npz', allow_pickle=False)
ids, lengths = z['ids'], z['lengths']
offsets = np.concatenate([[0], np.cumsum(lengths)])
id2idx = {c: i for i, c in enumerate(ids)}
frames = np.load(PROC / 'frames_tracked_v2.npy', mmap_mode='r')
assert len(frames) == offsets[-1], 'frames_tracked_v2.npy does not match landmarks_raw.npz: re-run Stage 0 cells 10 and 11b.'

meta = splits[splits['in_subset']].reset_index(drop=True)
y_all = meta['label_id_subset'].astype(int).to_numpy()
N_CLASSES = int(y_all.max()) + 1
assert len(np.unique(y_all)) == N_CLASSES

tr = (meta['split'] == 'train').to_numpy(); va = (meta['split'] == 'val').to_numpy(); te = (meta['split'] == 'test').to_numpy()
folds = meta['fold'].to_numpy()
print(f'{len(meta)} clips | {N_CLASSES} signs | chance {1 / N_CLASSES:.2%} | train {tr.sum()} val {va.sum()} test {te.sum()}')

In [ ]:
# [variants] Cell 4 — build the sequence tensors (64 frames x 132 numbers) for the four normalization variants (cached)
from tqdm.auto import tqdm
WRIST, MIDDLE_MCP, PALM = 0, 9, [5, 9, 13, 17]          # wrist, middle finger base, and the four finger bases (index, middle, ring, pinky)

def normalize_hands_v(x, aspect, variant):
    """x: (..., 21, 3) raw landmarks. Wrist-centred hand shape divided by a hand-size scale; NaN where the hand is missing."""
    x = x.astype(np.float32).copy()
    x[..., 0] *= aspect                                        # x: fraction of width -> height units
    robust = variant.startswith('robust')
    if robust:
        x[..., 2] *= aspect                                    # z uses about the same scale as x (width units) -> height units
    centred = x - x[..., WRIST:WRIST + 1, :]
    if robust:
        scale = np.linalg.norm(centred[..., PALM, :], axis=-1).mean(-1)          # mean 3D length of four palm bones
    else:
        scale = np.linalg.norm(centred[..., MIDDLE_MCP, :2], axis=-1)            # one projected bone (the current choice)
    scale = np.where(scale < 1e-6, np.nan, scale)
    return centred / scale[..., None, None]

def clip_to_sequence_v(clip, aspect, variant, T=FIXED_T):
    """(L, 2, 21, 3) -> (T, 132): 126 joint values (hand 0 then hand 1), 4 wrist-position values, 2 visibility flags."""
    L = len(clip)
    idx = np.linspace(0, L - 1, T).round().astype(int)
    hands = normalize_hands_v(clip, aspect, variant)[idx]
    wrist = clip[:, :, WRIST, :2].astype(np.float32).copy()
    wrist[..., 0] *= aspect
    wrist = wrist[idx]
    present = ~np.isnan(hands).all(axis=(2, 3))
    hands = np.where(np.isnan(hands), 0.0, hands)
    if variant.endswith('clip5'):
        hands = np.clip(hands, -5.0, 5.0)
    wrist = np.where(present[:, :, None], wrist, 0.0)
    return np.concatenate([hands.reshape(T, 126), wrist.reshape(T, 4), present.astype(np.float32)], axis=1).astype(np.float32)

# --- unit tests on a synthetic clip ---
_rng = np.random.default_rng(0)
_clip = _rng.normal(size=(40, 2, 21, 3)).astype(np.float32); _clip[10:15, 1] = np.nan
for v in VARIANTS:
    s = clip_to_sequence_v(_clip, 1.0, v)
    assert s.shape == (FIXED_T, 132) and np.isfinite(s).all(), v
    moved = clip_to_sequence_v(_clip * 2.0 + np.array([5.0, -3.0, 4.0], np.float32), 1.0, v)
    assert np.allclose(s[:, :126], moved[:, :126], atol=1e-4), f'{v}: joint channels must not depend on hand position or size'
    if v.endswith('clip5'):
        assert np.abs(s[:, :126]).max() <= 5.0
print('OK: variant unit tests passed')

meta_ids = meta['attachment_id'].to_numpy().astype(str)
SEQV = {}
for v in VARIANTS:
    f = PROC / (f'sequences_subset_T{FIXED_T}.npz' if v == 'base' else f'sequences_subset_T{FIXED_T}_{v}.npz')
    arr = None
    if f.exists():
        zs = np.load(f, allow_pickle=False)
        if zs['ids'].shape == meta_ids.shape and (zs['ids'] == meta_ids).all():
            arr = zs['S'].astype(np.float32); print(f'{v:15s} loaded from cache {f.name}')
    if arr is None:
        arr = np.zeros((len(meta), FIXED_T, 132), np.float32)
        for r, (cid, asp) in enumerate(tqdm(zip(meta['attachment_id'], meta['aspect']), total=len(meta), desc=v)):
            i = id2idx[cid]
            arr[r] = clip_to_sequence_v(np.asarray(frames[offsets[i]:offsets[i + 1]], dtype=np.float32), float(asp), v)
        np.savez_compressed(f, S=arr.astype(np.float16), ids=meta_ids)
        print(f'{v:15s} built and saved {f.name}')
    assert arr.shape == (len(meta), FIXED_T, 132) and np.isfinite(arr).all()
    SEQV[v] = arr
SEQ = SEQV['base']                                                  # the augmentation unit tests below use this one

In [ ]:
# Cell 5 — how heavy are the tails in each variant? (visible hands, training clips)
def tail_stats(S):
    vis = S[:, :, 130:132].astype(bool)
    vals = np.abs(S[:, :, :126].reshape(len(S), FIXED_T, 2, 63)[vis])
    return {**{f'p{p}': float(np.percentile(vals, p)) for p in (50, 90, 99, 99.9)}, 'max': float(vals.max()),
            '>5': float((vals > 5).mean()), '>10': float((vals > 10).mean())}
tails = pd.DataFrame({v: tail_stats(SEQV[v][tr]) for v in VARIANTS}).T
print(tails.round(4).to_string())

# is the tail caused by tiny hand-size scales in the current normalization?  (1,500 training clips)
sub_idx = np.random.default_rng(0).choice(np.where(tr)[0], min(1500, int(tr.sum())), replace=False)
sc, big = [], []
for r in sub_idx:
    i = id2idx[meta['attachment_id'].iloc[r]]
    clip = np.asarray(frames[offsets[i]:offsets[i + 1]], dtype=np.float32) * np.array([meta['aspect'].iloc[r], 1.0, 1.0], np.float32)
    cen = clip - clip[:, :, WRIST:WRIST + 1, :]
    s = np.linalg.norm(cen[:, :, MIDDLE_MCP, :2], axis=-1)                 # (L, 2) the current scale
    v = np.abs(cen[..., :2] / np.maximum(s, 1e-6)[..., None, None]).max(axis=(-1, -2))       # largest |x| or |y| in the frame, (L, 2)
    ok = ~np.isnan(s); sc.append(s[ok]); big.append(v[ok])
sc, big = np.concatenate(sc), np.concatenate(big)
q = np.percentile(sc, [1, 5, 50, 95])
print(f'\ncurrent hand-size scale (image-height units) percentiles 1/5/50/95: {q.round(4)}')
small = sc < q[1]
print(f'frames with the 5 % smallest scales contain {(big[small] > 10).sum() / max((big > 10).sum(), 1):.0%} of the frames that have a value above 10 (random would be 5 %)')

In [ ]:
# [aug] Cell 6 — the augmentations (numpy, applied to RAW normalized hand shapes) and their unit tests
def seq_mask(S):
    """(N, T, 132) mask: 1 where the channel's hand is visible (or the channel is a visibility flag)."""
    p = S[:, :, 130:132]
    return np.concatenate([np.repeat(p[:, :, :1], 63, 2), np.repeat(p[:, :, 1:], 63, 2),
                           np.repeat(p[:, :, :1], 2, 2), np.repeat(p[:, :, 1:], 2, 2), np.ones_like(p)], axis=2)

def seq_stats(S_tr):
    m = seq_mask(S_tr)
    cnt = np.maximum(m.sum((0, 1)), 1.0)
    mean = (S_tr * m).sum((0, 1)) / cnt
    std = np.sqrt((((S_tr - mean) ** 2) * m).sum((0, 1)) / cnt)
    std = np.where(std < 1e-3, 1.0, std)
    mean[130:], std[130:] = 0.0, 1.0
    return mean.astype(np.float32), std.astype(np.float32)

def apply_seq_norm(S, mean, std):
    return (((S - mean) / std) * seq_mask(S)).astype(np.float32)           # missing hands stay exactly 0

BASE_MAG = {'rotation': 15.0, 'scale': 0.15, 'timewarp': 0.20, 'noise': 0.03,
            'framehold': 0.10, 'jointdrop': 0.05, 'wristjitter': 0.05}
ORDER = ['timewarp', 'framehold', 'rotation', 'scale', 'noise', 'jointdrop', 'wristjitter']
P_APPLY = 0.5                                                              # each augmentation is applied to a clip with this probability

def _joints(x):
    return x[:, :, :126].reshape(len(x), x.shape[1], 2, 21, 3)

def _put(x, j):
    out = x.copy()
    out[:, :, :126] = j.reshape(len(x), x.shape[1], 126)
    return out

def _on(rng, B):
    return (rng.random(B) < P_APPLY).astype(np.float32)

def aug_rotation(x, rng, mag):                                             # mag = maximum angle in degrees
    B = len(x)
    th = np.deg2rad(rng.uniform(-mag, mag, B)) * _on(rng, B)
    c = np.cos(th).reshape(B, 1, 1, 1).astype(np.float32); s = np.sin(th).reshape(B, 1, 1, 1).astype(np.float32)
    j = _joints(x).copy(); xx, yy = j[..., 0].copy(), j[..., 1].copy()
    j[..., 0] = c * xx - s * yy
    j[..., 1] = s * xx + c * yy
    return _put(x, j)

def aug_scale(x, rng, mag):                                                # mag = maximum relative change
    B = len(x)
    f = (1 + rng.uniform(-mag, mag, B) * _on(rng, B)).astype(np.float32).reshape(B, 1, 1, 1, 1)
    return _put(x, _joints(x) * f)

def aug_timewarp(x, rng, mag):                                             # mag = maximum relative speed change (shift = mag * 20 frames)
    B, T = x.shape[:2]
    on = _on(rng, B)
    r = 1 + rng.uniform(-mag, mag, B) * on
    sh = rng.uniform(-mag, mag, B) * 20 * on
    t, c = np.arange(T)[None, :], (T - 1) / 2
    idx = np.clip(np.rint(c + (t - c) * r[:, None] + sh[:, None]), 0, T - 1).astype(int)
    return np.take_along_axis(x, idx[:, :, None], axis=1)

def aug_framehold(x, rng, mag):                                            # mag = probability that a frame repeats the previous one
    B, T = x.shape[:2]
    drop = (rng.random((B, T)) < min(mag, 0.5)) & (_on(rng, B)[:, None] > 0)
    drop[:, 0] = False
    idx = np.maximum.accumulate(np.where(drop, 0, np.arange(T)[None, :]), axis=1)    # last kept frame
    return np.take_along_axis(x, idx[:, :, None], axis=1)

def aug_noise(x, rng, mag):                                                # mag = standard deviation, visible hands only
    B, T = x.shape[:2]
    vis = x[:, :, 130:132].reshape(B, T, 2, 1, 1)
    n = rng.normal(0, mag, (B, T, 2, 21, 3)).astype(np.float32) * vis * _on(rng, B).reshape(B, 1, 1, 1, 1)
    return _put(x, _joints(x) + n)

def aug_jointdrop(x, rng, mag):                                            # mag = probability that a joint is set to 0
    B, T = x.shape[:2]
    m = (rng.random((B, T, 2, 21, 1)) < min(mag, 0.5)) & (_on(rng, B).reshape(B, 1, 1, 1, 1) > 0)
    return _put(x, np.where(m, 0.0, _joints(x)).astype(np.float32))

def aug_wristjitter(x, rng, mag):                                          # mag = standard deviation of a constant shift, visible hands only
    B, T = x.shape[:2]
    vis = x[:, :, 130:132].reshape(B, T, 2, 1)
    off = rng.normal(0, mag, (B, 1, 2, 2)).astype(np.float32) * _on(rng, B).reshape(B, 1, 1, 1)
    out = x.copy()
    out[:, :, 126:130] = (x[:, :, 126:130].reshape(B, T, 2, 2) + off * vis).reshape(B, T, 4)
    return out

AUG_FUNCS = {'rotation': aug_rotation, 'scale': aug_scale, 'timewarp': aug_timewarp, 'noise': aug_noise,
             'framehold': aug_framehold, 'jointdrop': aug_jointdrop, 'wristjitter': aug_wristjitter}

def augment(x, names, rng, strength=1.0):
    """Apply the named augmentations (always in the same order) to a batch x of RAW sequences (B, T, 132)."""
    for name in ORDER:
        if name in names:
            x = AUG_FUNCS[name](x, rng, BASE_MAG[name] * strength)
    return x

# --- unit tests -------------------------------------------------------------------------------------------------------
_rng = np.random.default_rng(0)
_x = SEQ[tr][:32].copy()

def _consistent(o):
    """Wherever a hand's visibility flag is 0, that hand's joints and wrist position must be exactly 0."""
    B, T = o.shape[:2]
    inv = 1.0 - o[:, :, 130:132]
    return (np.abs(o[:, :, :126].reshape(B, T, 2, 63)) * inv[..., None]).max() == 0 and \
           (np.abs(o[:, :, 126:130].reshape(B, T, 2, 2)) * inv[..., None]).max() == 0

assert _consistent(_x), 'the input data itself is inconsistent'
for name in ORDER:
    out = augment(_x, (name,), _rng)
    assert out.shape == _x.shape and out.dtype == np.float32 and np.isfinite(out).all(), name
    assert _consistent(out), f'{name}: a hand that is flagged missing has non-zero values'
    if name not in ('timewarp', 'framehold'):                                  # these two only reorder frames in time
        assert np.array_equal(out[:, :, 130:132], _x[:, :, 130:132]), f'{name} changed the visibility flags'
    assert not np.array_equal(out, _x) or name in ('timewarp', 'framehold'), f'{name} did nothing'
    zero = AUG_FUNCS[name](_x, _rng, 0.0)                                      # magnitude 0 = identity
    assert np.allclose(zero, _x, atol=1e-6), f'{name} with magnitude 0 is not the identity'
# rotation keeps distances; framehold repeats the previous frame; timewarp at speed 1 and shift 0 is the identity
P_APPLY_BACKUP, P_APPLY = P_APPLY, 1.0
r = aug_rotation(_x, _rng, 30.0)
assert np.allclose(np.linalg.norm(_joints(r)[..., :2], axis=-1), np.linalg.norm(_joints(_x)[..., :2], axis=-1), atol=1e-4), 'rotation changed distances'
h = aug_framehold(_x, _rng, 0.3)
for b in range(len(h)):
    for t in range(1, h.shape[1]):
        assert np.array_equal(h[b, t], _x[b, t]) or np.array_equal(h[b, t], h[b, t - 1]), 'framehold produced a frame that is neither new nor repeated'
P_APPLY = P_APPLY_BACKUP
print('OK: augmentation unit tests passed for', ORDER)

In [ ]:
# [torch] Cell 7 — the two networks (CNN and GRU, both take (batch, time, channels)) and the training loop with augmentation
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class CNN1D(nn.Module):
    """Stage 3 network: blocks of [Conv1d -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout], global average + max pooling, Linear."""
    def __init__(self, in_ch, n_classes, channels=(128, 128, 256), kernel=9, dropout=0.5, block_dropout=0.1):
        super().__init__()
        blocks, c_in = [], in_ch
        for c_out in channels:
            blocks += [nn.Conv1d(c_in, c_out, kernel, padding=kernel // 2), nn.BatchNorm1d(c_out), nn.ReLU(),
                       nn.MaxPool1d(2), nn.Dropout(block_dropout)]
            c_in = c_out
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * c_in, n_classes))

    def forward(self, x):                                         # x: (batch, time, channels)
        h = self.features(x.transpose(1, 2))
        h = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)
        return self.head(h)

class GRUNet(nn.Module):
    """Stage 4 network: optional conv front-end, (Bi)GRU, pooling over time, Linear. Input (batch, time, channels)."""
    def __init__(self, in_ch, n_classes, hidden=256, layers=2, bidirectional=True, dropout=0.5, rnn_dropout=0.3,
                 pooling='meanmax', frontend='conv', conv_ch=128):
        super().__init__()
        if frontend == 'conv':
            self.frontend = nn.Sequential(nn.Conv1d(in_ch, conv_ch, 5, padding=2), nn.BatchNorm1d(conv_ch), nn.ReLU())
            rnn_in = conv_ch
        else:
            self.frontend, rnn_in = None, in_ch
        self.rnn = nn.GRU(rnn_in, hidden, num_layers=layers, batch_first=True, bidirectional=bidirectional,
                          dropout=rnn_dropout if layers > 1 else 0.0)
        d = hidden * (2 if bidirectional else 1)
        self.pooling, self.bidirectional = pooling, bidirectional
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * d if pooling == 'meanmax' else d, n_classes))

    def forward(self, x):
        if self.frontend is not None:
            x = self.frontend(x.transpose(1, 2)).transpose(1, 2)
        out, h_n = self.rnn(x)
        if self.pooling == 'meanmax':
            h = torch.cat([out.mean(dim=1), out.amax(dim=1)], dim=1)
        elif self.bidirectional:
            h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h = h_n[-1]
        return self.head(h)

def new_model(cfg, in_ch, seed):
    set_seed(seed)
    if cfg['kind'] == 'cnn':
        m = CNN1D(in_ch, N_CLASSES, cfg['channels'], cfg['kernel'], cfg['dropout'], cfg['block_dropout'])
    else:
        m = GRUNet(in_ch, N_CLASSES, cfg['hidden'], cfg['layers'], cfg['bidirectional'], cfg['dropout'], cfg['rnn_dropout'],
                   cfg['pooling'], cfg['frontend'], cfg['conv_ch'])
    return m.to(DEVICE)

@torch.no_grad()
def check_architecture(cfg, in_ch=132):
    """Dummy batch through the network: returns (output shape, number of parameters)."""
    m = new_model(cfg, in_ch, 0).eval()
    out = m(torch.zeros(2, FIXED_T, in_ch, device=DEVICE))
    return tuple(out.shape), sum(p.numel() for p in m.parameters())

@torch.no_grad()
def _eval(model, X, y):
    model.eval()
    logits = model(X)
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

def fit(model, Xtr_raw, ytr, mean, std, Xva=None, yva=None, aug=(), strength=1.0, epochs=200, lr=1e-3, weight_decay=1e-2,
        batch_size=128, patience=25, label_smoothing=0.1, seed=0, fixed_epochs=None, grad_clip=0.0):
    """Mini-batch training as in Stages 2 to 4. Xtr_raw are the RAW training sequences (N, T, 132); when `aug` is not empty,
    every mini-batch is augmented on the fly (numpy) and then standardized with the TRAIN mean/std. Xva is already standardized."""
    t0 = time.time()
    n_epochs = fixed_epochs or epochs
    g = torch.Generator(); g.manual_seed(seed)                              # order of the mini-batches
    arng = np.random.default_rng(10_000 + seed)                             # randomness of the augmentation
    Xclean = torch.as_tensor(apply_seq_norm(Xtr_raw, mean, std), dtype=torch.float32, device=DEVICE)   # for the training-accuracy curve
    yt = torch.as_tensor(ytr, dtype=torch.long, device=DEVICE)
    use_val = (Xva is not None) and (fixed_epochs is None)
    if use_val:
        Xv = torch.as_tensor(Xva, dtype=torch.float32, device=DEVICE)
        yv = torch.as_tensor(yva, dtype=torch.long, device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_epoch, best_state, bad, n = -1.0, 0, None, 0, len(Xtr_raw)
    for epoch in range(1, n_epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=g)                              # CPU tensor
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            if aug:
                xb = torch.as_tensor(apply_seq_norm(augment(Xtr_raw[idx.numpy()], aug, arng, strength), mean, std),
                                     dtype=torch.float32, device=DEVICE)
            else:
                xb = Xclean[idx.to(DEVICE)]
            yb = yt[idx.to(DEVICE)]
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
        tl, ta = _eval(model, Xclean, yt)
        hist['train_loss'].append(tl); hist['train_acc'].append(ta)
        if use_val:
            vl, vacc = _eval(model, Xv, yv)
            hist['val_loss'].append(vl); hist['val_acc'].append(vacc)
            if vacc > best_acc:
                best_acc, best_epoch, bad = vacc, epoch, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if use_val and best_state is not None:
        model.load_state_dict(best_state)
    else:
        best_epoch = n_epochs
    return hist, best_epoch, time.time() - t0

@torch.no_grad()
def predict_logits(model, X, batch_size=512):
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        xb = torch.as_tensor(X[i:i + batch_size], dtype=torch.float32, device=DEVICE)
        out.append(model(xb).cpu().numpy())
    return np.concatenate(out)

print('networks and training loop defined')

In [ ]:
# [runner] Cell 8 — configurations, metrics and the experiment runner
from sklearn.metrics import f1_score

CNN_CFG = dict(kind='cnn', channels=(128, 128, 256), kernel=9, dropout=0.5, block_dropout=0.1, grad_clip=0.0)   # Stage 3 final
GRU_CFG = dict(kind='gru', hidden=256, layers=2, bidirectional=True, dropout=0.5, rnn_dropout=0.3, pooling='meanmax',
               frontend='conv', conv_ch=128, grad_clip=1.0)                                                      # Stage 4 final
TRAIN_CFG = dict(lr=1e-3, weight_decay=1e-2, batch_size=128, epochs=200, patience=25, label_smoothing=0.1)      # more room than before: augmentation converges slower

for nm, cfg in (('CNN', CNN_CFG),):
    shape, n_par = check_architecture(cfg)
    assert shape == (2, N_CLASSES), f'unexpected output shape {shape}'
    print(f'{nm}: output {shape} | {n_par:,} parameters')

def metrics_from_logits(logits, y):
    pred = logits.argmax(1)
    top5 = np.argsort(-logits, axis=1)[:, :5]
    return {'n': int(len(y)), 'top1': float((pred == y).mean()), 'top5': float(np.mean([yt in t for yt, t in zip(y, top5)])),
            'macro_f1': float(f1_score(y, pred, average='macro')), 'pred': pred, 'logits': logits.astype(np.float32)}

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

MASKS = {'train': tr, 'val': va, 'test': te}
results = []
_cache = {}

def mask_key(masks):
    return tuple((k, hash(m.tobytes())) for k, m in masks.items())

def run_one(experiment, name, variant, model_cfg, aug, strength, masks, seed, fixed_epochs=None, use_cache=True):
    """Train one model on masks['train'] (augmenting if `aug` is not empty), evaluate on every other mask."""
    ckey = (variant, json.dumps(model_cfg, sort_keys=True, default=list), tuple(aug), float(strength), seed, mask_key(masks), fixed_epochs)
    if use_cache and ckey in _cache:
        return _cache[ckey]
    S = SEQV[variant]
    Xtr = S[masks['train']]
    mean, std = seq_stats(Xtr)
    ev = {k: apply_seq_norm(S[m], mean, std) for k, m in masks.items() if k != 'train'}
    model = new_model(model_cfg, 132, seed)
    hist, best_ep, fit_s = fit(model, Xtr, y_all[masks['train']], mean, std, ev.get('val'), y_all[masks['val']] if 'val' in masks else None,
                               aug=tuple(aug), strength=strength, seed=seed, fixed_epochs=fixed_epochs,
                               grad_clip=model_cfg['grad_clip'], **TRAIN_CFG)
    out = {'name': name, 'seed': seed, 'hist': hist, 'best_epoch': best_ep, 'fit_s': fit_s, 'metrics': {}}
    for sname, m in masks.items():
        if sname == 'train':
            continue
        r = metrics_from_logits(predict_logits(model, ev[sname]), y_all[m])
        out['metrics'][sname] = r
        results.append({'experiment': experiment, 'name': name, 'variant': variant, 'aug': '+'.join(aug) or 'none', 'strength': strength, 'seed': seed,
                        'set': sname, 'n': r['n'], 'top1': r['top1'], 'top5': r['top5'], 'macro_f1': r['macro_f1'],
                        'best_epoch': best_ep, 'fit_s': fit_s})
    del model
    if use_cache:
        _cache[ckey] = out
    return out

def val_of(outs):
    v = [o['metrics']['val']['top1'] for o in outs]
    return float(np.mean(v)), float(np.std(v))

def summary_line(o, sets=('val', 'test')):
    return ' | '.join(f'{s} top-1 {o["metrics"][s]["top1"]:.1%} top-5 {o["metrics"][s]["top5"]:.1%}' for s in sets if s in o['metrics']) \
           + f' | best epoch {o["best_epoch"]} | {o["fit_s"]:.0f}s'

def save_results(tag=''):
    pd.DataFrame(results).to_csv(RES / 'stage4c_results.csv', index=False)

In [ ]:
# Cell 9 — the augmentation recipe chosen in Stage 4b (rotation, strength 1.0) and the sanity check
rf = RES / 'stage4b_recipe.json'
if rf.exists():
    rj = json.load(open(rf)); RECIPE, STRENGTH = tuple(rj['recipe']), float(rj['strength'])
else:
    RECIPE, STRENGTH = ('rotation',), 1.0
print('recipe:', RECIPE, 'strength', STRENGTH)

Xtr = SEQ[tr]; mean, std = seq_stats(Xtr)
small_i = np.random.default_rng(0).choice(len(Xtr), 64, replace=False)
cfg0 = {**CNN_CFG, 'dropout': 0.0, 'block_dropout': 0.0}
mdl = new_model(cfg0, 132, 0)
h, _, _ = fit(mdl, Xtr[small_i], y_all[tr][small_i], mean, std, aug=RECIPE, strength=0.5, fixed_epochs=200,
              lr=3e-3, weight_decay=0.0, batch_size=64, label_smoothing=0.0, seed=0)
print(f'train loss {h["train_loss"][0]:.2f} -> {h["train_loss"][-1]:.3f} | clean train accuracy {h["train_acc"][0]:.0%} -> {h["train_acc"][-1]:.0%}')
assert h['train_acc'][-1] > 0.8, 'The CNN cannot memorize 64 clips: something is wrong (send me this output).'
print('OK: the training pipeline works')
del mdl, Xtr

In [ ]:
# Cell 10 — Experiment 1: the CNN on each variant, 3 seeds, single split (training with the Stage 4b recipe)
t0 = time.time()
single = {}
for v in VARIANTS:
    runs = [run_one('V1 variants 3 seeds', f'CNN {v}', v, CNN_CFG, RECIPE, STRENGTH, MASKS, s, use_cache=False) for s in FINAL_SEEDS]
    single[v] = runs
    va_ = [o['metrics']['val']['top1'] for o in runs]; te_ = [o['metrics']['test']['top1'] for o in runs]; t5 = [o['metrics']['test']['top5'] for o in runs]
    print(f'{v:15s} val {np.mean(va_):.1%} +/- {np.std(va_):.1%} | test {np.mean(te_):.1%} +/- {np.std(te_):.1%} | test top-5 {np.mean(t5):.1%} | best epochs {[o["best_epoch"] for o in runs]} | {np.mean([o["fit_s"] for o in runs]):.0f}s per run')
    save_results()
print(f'{(time.time() - t0) / 60:.0f} min')
best_v = max(VARIANTS, key=lambda v: np.mean([o['metrics']['val']['top1'] for o in single[v]]))
print('best variant on VALIDATION (3 seeds):', best_v)

In [ ]:
# Cell 11 — Experiment 2: 5-fold cross-validation for every variant (same folds as Stages 1 to 4b) and paired comparisons
def cv_variant(v):
    acc = []
    for k in range(N_FOLDS):
        te_k = folds == k; va_k = folds == (k + 1) % N_FOLDS; tr_k = ~(te_k | va_k)
        o = run_one('V2 variants 5-fold CV', f'CNN {v}', v, CNN_CFG, RECIPE, STRENGTH, {'train': tr_k, 'val': va_k, 'test': te_k}, 0, use_cache=False)
        results[-2]['set'], results[-1]['set'] = f'fold {k} val', f'fold {k}'
        acc.append(o['metrics']['test']['top1'])
    acc = np.array(acc)
    print(f'{v:15s} folds ' + ', '.join(f'{a:.1%}' for a in acc) + f' | mean {acc.mean():.1%} +/- {acc.std():.1%}')
    save_results()
    return acc

CVV = {v: cv_variant(v) for v in VARIANTS}

print('\nPaired comparison with the current normalization (base), same folds:')
for v in VARIANTS[1:]:
    d = CVV[v] - CVV['base']
    print(f'  {v:15s} better in {(d > 0).sum()} of {N_FOLDS} folds | mean difference {d.mean():+.1%} (std error {d.std(ddof=1) / np.sqrt(N_FOLDS):.1%}; per fold: ' + ', '.join(f'{x:+.1%}' for x in d) + ')')

b4 = RES / 'stage4b_results.csv'
if b4.exists():
    d4 = pd.read_csv(b4)
    r = d4[(d4.experiment == 'F2 CNN 5-fold CV') & (d4.name == 'CNN + aug') & d4.set.str.fullmatch(r'fold \d')].sort_values('set').top1.to_numpy()
    if len(r) == N_FOLDS:
        print(f'\n(reference: Stage 4b CNN + rotation, same settings: {r.mean():.1%} +/- {r.std():.1%}; folds ' + ', '.join(f'{x:.1%}' for x in r) + ')')

In [ ]:
# Cell 12 — results table and figure
res = pd.DataFrame(results)
res.to_csv(RES / 'stage4c_results.csv', index=False)
print(res[res.experiment.str.startswith('V')].groupby(['experiment', 'variant', 'set'])[['top1', 'top5', 'macro_f1']].mean().round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
x = np.arange(len(VARIANTS))
ax[0].bar(x, [CVV[v].mean() for v in VARIANTS], yerr=[CVV[v].std() for v in VARIANTS], capsize=4, color=['#999'] + ['#4C78A8'] * (len(VARIANTS) - 1))
ax[0].set_xticks(x); ax[0].set_xticklabels(VARIANTS, rotation=20, ha='right'); ax[0].set_ylim(0.5, 0.75)
ax[0].set_ylabel('top-1 (5-fold CV, mean +/- std)'); ax[0].set_title('Normalization variants')
for v in VARIANTS:
    ax[1].plot(range(N_FOLDS), CVV[v], marker='o', label=v)
ax[1].set_xlabel('fold'); ax[1].set_title('Per fold (same test clips)'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / 'stage4c_results.png', dpi=150); plt.show()
print('Saved', RES / 'stage4c_results.csv', 'and', FIG / 'stage4c_results.png')

## Stage 4c checklist

- [ ] Cell 4: `OK: variant unit tests passed` and the four variants built or loaded
- [ ] Cell 5: tail table (percentiles, share above 5 and 10) for the four variants
- [ ] Cell 9: `OK: the training pipeline works`
- [ ] Cell 10: 3 seeds per variant on the single split
- [ ] Cell 11: 5-fold CV per variant and the paired comparisons with `base`
- [ ] `results/stage4c_results.csv` and `figures/stage4c_results.png` exist

**Send back:** the printed outputs of Cells 4, 5, 9, 10 and 11 and the figure `stage4c_results.png`.
If a cell fails, send me the last lines of the error.